In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import seaborn as sns
import plotly.graph_objects as go
from plotly.offline import iplot

In [ ]:
train = None
with open("../../Data/PHM2025_training_data/training_data.csv", "r") as f:
    train = pd.read_csv(f)

train.head()

In [ ]:
esn101 = train[train['ESN'] == 101 ]
N = len(esn101) # Get the number of rows
esn101['i'] = pd.RangeIndex(start=0, stop=N)

esn102 = train[train['ESN'] == 102 ]
N = len(esn102) # Get the number of rows
esn102['i'] = pd.RangeIndex(start=0, stop=N)

esn103 = train[train['ESN'] == 103 ]
N = len(esn103) # Get the number of rows
esn103['i'] = pd.RangeIndex(start=0, stop=N)

esn104 = train[train['ESN'] == 104 ]
N = len(esn104) # Get the number of rows
esn104['i'] = pd.RangeIndex(start=0, stop=N)


train_2 = pd.concat([esn101, esn102, esn103, esn104], axis=0)

In [ ]:
sensors = ["Sensed_Altitude", "Sensed_Mach", "Sensed_Pamb", "Sensed_Pt2", "Sensed_TAT", "Sensed_WFuel",
           "Sensed_VAFN", "Sensed_VBV", "Sensed_Fan_Speed", "Sensed_Core_Speed", "Sensed_T25",
           "Sensed_T3", "Sensed_Ps3", "Sensed_T45", "Sensed_P25", "Sensed_T5"]

def plot(df, sensor):
  fig = px.line(
        df,
        x=df['i'],
        y=sensor,
        color='ESN',
        title=f'Andamento del Sensore {sensor} Suddiviso per ESN',
        labels={'x':'Indice del Dato / Tempo (Sequenza)', 'y':f'Valore di {sensor}'},
        height=800,
        line_group='ESN' # Garantisce che i punti siano collegati correttamente per ogni gruppo ESN
    )

  # Aggiorna il layout per un aspetto migliore (opzionale)
  fig.update_xaxes(rangeslider_visible=True) # Aggiunge uno slider in basso per navigare nel tempo

  # Mostra il grafico interattivo in Colab/Jupyter
  fig.show()

In [ ]:
plot(train_2[train_2['ESN'] == 101], sensors[5])

In [ ]:
cm = train.corr()

plt.figure(figsize=(15,15))
sns.heatmap(cm, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix Heatmap')
plt.show()

In [ ]:
# Assuming your DataFrame is 'df' and the column is 'sensor_value'
wws_points = train[train['Cumulative_WWs'] > train['Cumulative_WWs'].shift(1)]
hpc_points = train[train['Cumulative_HPC_SVs'] > train['Cumulative_HPC_SVs'].shift(1)]
hpt_points = train[train['Cumulative_HPT_SVs'] > train['Cumulative_HPT_SVs'].shift(1)]

In [ ]:
wws_points

In [ ]:
# # Identificazione degli eventi di manutenzione (cambio nei contatori cumulativi)
# event_data = train.copy()

# # Calcola la differenza per identificare i cicli in cui i contatori cumulativi cambiano
# event_data['WWs_Change'] = event_data.groupby('ESN')['Cumulative_WWs'].diff().fillna(0)
# event_data['HPC_SVs_Change'] = event_data.groupby('ESN')['Cumulative_HPC_SVs'].diff().fillna(0)
# event_data['HPT_SVs_Change'] = event_data.groupby('ESN')['Cumulative_HPT_SVs'].diff().fillna(0)

# # Filtra per trovare solo i cicli in cui si è verificato un evento (cambio > 0)
# # Questi saranno usati per disegnare le linee verticali.
# wws_events = event_data[event_data['WWs_Change'] > 0]
# hpc_events = event_data[event_data['HPC_SVs_Change'] > 0]
# hpt_events = event_data[event_data['HPT_SVs_Change'] > 0]

# # Elenco dei sensori da plottare (tutte le colonne numeriche escluse le colonne ID/Cumulative)
# id_cols = ['ESN', 'Cycles_Since_New', 'Snapshot', 'Cycles_to_WW', 'Cycles_to_HPC_SV', 'Cycles_to_HPT_SV']
# cumulative_cols = ['Cumulative_WWs', 'Cumulative_HPC_SVs', 'Cumulative_HPT_SVs']
# change_cols = ['WWs_Change', 'HPC_SVs_Change', 'HPT_SVs_Change']

# sensor_cols = [col for col in train.columns if col not in id_cols + cumulative_cols + change_cols]
# print("\nColonne Sensore disponibili per il plot:")
# print(sensor_cols)

# # Colonna sensore di default
# DEFAULT_SENSOR = 'Sensed_T25'

In [ ]:
# Mappatura dei colori per gli eventi di manutenzione
EVENT_COLORS = {
    'Cumulative_WWs': 'red',
    'Cumulative_HPC_SVs': 'blue',
    'Cumulative_HPT_SVs': 'green'
}

def create_figure(df, sensor_name, wws_df, hpc_df, hpt_df, esn):
    """Crea una figura Plotly per il sensore specificato."""
    
    df = df[df["ESN"] == esn]
    wws_df = wws_df[wws_df["ESN"] == esn]
    hpc_df = hpc_df[hpc_df["ESN"] == esn]
    hpt_df = hpt_df[hpt_df["ESN"] == esn]
    
    fig = px.line(
        df,
        x=df['i'],
        y=sensor_name,
        color='ESN',
        title=f'Andamento del Sensore {sensor_name} Suddiviso per ESN',
        labels={'x':'Indice del Dato / Tempo (Sequenza)', 'y':f'Valore di {sensor_name}'},
        height=800,
        line_group='ESN'
    )
    
    for index, row in wws_df.iterrows():
        fig.add_vline(
            x=row['Cycles_Since_New'],
            line_width=1, line_dash="dash", line_color=EVENT_COLORS['Cumulative_WWs'],
            name='WWs Event',
            # Aggiunge un'annotazione per identificare il tipo di evento
            annotation_text="WWs", annotation_position="top right",
            annotation_font_color=EVENT_COLORS['Cumulative_WWs']
        )
    
    # Eventi Cumulative_HPC_SVs (Manutenzione HPC)
    for index, row in hpc_df.iterrows():
        fig.add_vline(
            x=row['Cycles_Since_New'],
            line_width=1, line_dash="dash", line_color=EVENT_COLORS['Cumulative_HPC_SVs'],
            name='HPC_SV Event',
            annotation_text="HPC_SV", annotation_position="top left",
            annotation_font_color=EVENT_COLORS['Cumulative_HPC_SVs']
        )
        
    # Eventi Cumulative_HPT_SVs (Manutenzione HPT)
    for index, row in hpt_df.iterrows():
        fig.add_vline(
            x=row['Cycles_Since_New'],
            line_width=1, line_dash="dash", line_color=EVENT_COLORS['Cumulative_HPT_SVs'],
            name='HPT_SV Event',
            annotation_text="HPT_SV", annotation_position="bottom right",
            annotation_font_color=EVENT_COLORS['Cumulative_HPT_SVs']
        )


    fig.update_layout(
        title=f'Andamento del sensore "{sensor_name}" vs Cicli con Eventi di Manutenzione',
        xaxis_title='Cicli Dall\'Inizio (Cycles_Since_New)',
        yaxis_title=sensor_name,
        legend_title="ESN (Motore)",
        height=600,
        hovermode="x unified"
    )

    return fig


def update_plot(sensor_name, data, esn):
    return create_figure(data, sensor_name, wws_points, hpc_points, hpt_points, esn)

In [ ]:
for sensor in sensors:
    for esn in train["ESN"].unique():
        fig = update_plot(sensor, train_2, esn)
        iplot(fig)

In [ ]:
# Mappatura dei colori per gli eventi di manutenzione
EVENT_COLORS = {
    'Cumulative_WWs': 'red',
    'Cumulative_HPC_SVs': 'blue',
    'Cumulative_HPT_SVs': 'green'
}

# SENSOR_COLUMN di default (puoi cambiarlo qui)
DEFAULT_SENSOR = 'Sensed_T25' 

# La funzione è stata adattata per prendere in input un singolo ESN
def create_figure_for_esn(df, sensor_name, esn_value, wws_df, hpc_df, hpt_df):
    """Crea una figura Plotly per il sensore e l'ESN specificato."""
    
    # Filtra il DataFrame per l'ESN corrente
    engine_data = df[df['ESN'] == esn_value]
    
    # Filtra i DataFrame degli eventi per l'ESN corrente
    wws_esn = wws_df[wws_df['ESN'] == esn_value]
    hpc_esn = hpc_df[hpc_df['ESN'] == esn_value]
    hpt_esn = hpt_df[hpt_df['ESN'] == esn_value]

    fig = go.Figure()

    # 1. Plot della linea per l'ESN specifico
    fig.add_trace(go.Scatter(
        x=engine_data['Cycles_Since_New'],
        y=engine_data[sensor_name],
        mode='lines',
        name=f'ESN {esn_value} - {sensor_name}',
        # Colore fisso per la linea in questo caso (ad esempio, nero o grigio)
        line=dict(color='black', width=2), 
        hovertemplate='ESN: %{customdata[0]}<br>Cicli: %{x}<br>Valore: %{y}<extra></extra>',
        customdata=engine_data[['ESN', sensor_name]].values
    ))

    # 2. Aggiunta delle linee verticali per gli eventi
    
    # Eventi Cumulative_WWs (Washeo)
    for index, row in wws_esn.iterrows():
        fig.add_vline(
            x=row['Cycles_Since_New'],
            line_width=1, line_dash="dash", line_color=EVENT_COLORS['Cumulative_WWs'],
            name='WWs Event',
            annotation_text="WWs", annotation_position="top right",
            annotation_font_color=EVENT_COLORS['Cumulative_WWs']
        )
    
    # Eventi Cumulative_HPC_SVs (Manutenzione HPC)
    for index, row in hpc_esn.iterrows():
        fig.add_vline(
            x=row['Cycles_Since_New'],
            line_width=1, line_dash="dash", line_color=EVENT_COLORS['Cumulative_HPC_SVs'],
            name='HPC_SV Event',
            annotation_text="HPC_SV", annotation_position="top left",
            annotation_font_color=EVENT_COLORS['Cumulative_HPC_SVs']
        )
        
    # Eventi Cumulative_HPT_SVs (Manutenzione HPT)
    for index, row in hpt_esn.iterrows():
        fig.add_vline(
            x=row['Cycles_Since_New'],
            line_width=1, line_dash="dash", line_color=EVENT_COLORS['Cumulative_HPT_SVs'],
            name='HPT_SV Event',
            annotation_text="HPT_SV", annotation_position="bottom right",
            annotation_font_color=EVENT_COLORS['Cumulative_HPT_SVs']
        )


    fig.update_layout(
        title=f'ESN {esn_value} - Andamento del sensore "{sensor_name}" vs Cicli con Eventi di Manutenzione',
        xaxis_title='Cicli Dall\'Inizio (Cycles_Since_New)',
        yaxis_title=sensor_name,
        legend_title="Dati Sensore",
        height=400,
        hovermode="x unified",
        showlegend=False # La legenda non è necessaria con un solo motore
    )

    return fig

# --- Ciclo per generare e visualizzare tutti i grafici ---

# Verifica che il DataFrame 'train' sia disponibile prima di eseguire il ciclo
try:
    if 'train' not in locals():
        print("Errore: Il DataFrame 'train' non è disponibile. Caricalo prima di eseguire questo codice.")
    else:
        unique_esns = train['ESN'].unique()
        print(f"\nInizio la generazione di {len(unique_esns)} grafici per il sensore: {DEFAULT_SENSOR}...")
        
        # Generazione dei grafici
        for esn in unique_esns:
            fig = create_figure_for_esn(train, DEFAULT_SENSOR, esn, wws_points, hpc_points, hpt_points)
            iplot(fig) # Usa fig.show() in un ambiente Jupyter
            
        print("\nGenerazione grafici completata.")

except NameError:
    print("Errore: Variabili di pre-elaborazione (es. 'train', 'wws_events') non trovate. Assicurati di aver eseguito prima la sezione 'Caricamento Dati e Pre-elaborazione'.")